In [2]:
pip install transformers torch librosa jiwer

In [4]:
import torch
import time
import librosa
import re
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from jiwer import wer, cer


AUDIO_FILE = "out-0015592831-5053-20260802-125454-1785654594.80151.wav"

MODEL_ID = "sumanpaudel1997/nepali-asr-indicwav2vec"

REFERENCE_TEXT = """हेल्लो हजुर सुब्बाकार्यबाट बोल्नुभएको हो नि म्याम हैन हजुर हजुर एकछिन मैले अहिले चलाउँदै थिएँ एकछिन एक्सेप्ट गर्दिनु न है मेरो काम सकिएको छैन त्यही भएर ए एक्सेप्ट गरेँ अनि रोक्नुस् है हजुरले अहिलेको इयरमा नि चेक गर्नुभयो हैन हजुर हजुर एकचोटि लास्ट इयरमा लगिन गरेर मलाई देखाउनुस् न है लग अफ गर्ने लगिन लास्ट इयरमा हजुरको लास्ट इयरमा नि हजुर लास्ट इयरको लगिन गरेर देखाउनुस् न है यहाँबाट होला सायद विन्डोबाट हजुर लगिन अ त्यहाँबाट अब लास्ट इयरको लगिन गर्ने है अब ड्रप डाउन गर्नु सरले यही देखाउनुभएको थियो मलाई त्यही भएर एकचोटि तल टिक लगाउनु न यतापट्टि यतापट्टि ल फेरि कहाँ गयो अघि यसैमा गरेको हैन र अघि यसैमा गरेको हो अब हुनुपर्ने हो रजनीगन्धा देऊ त यसैमा हैन हजुर अब सेकेन्डमा जाने हजुर अब त्यहाँ ड्रप डाउनमा जाने अब ८२ ८३ देखाएको छैन है यो दुइटा २३० + २६५ एकछिन है म चेक गर्छु ल ल म होल्डमा राख्नु हो अझ तल तल गर्नु न देखाएको होला देखाएको छ नि म्याम अब ८२ ८३ मा जानु अनि लगिन गर्नु अब लगिन गर्नु लगिन सक्सेसफुल भयो है मलाई एकछिन है मैले यो एसक्यूएल बन्द गर्छु एकैछिन है अब अरु केही प्रोब्लम आयो भने कल गर्नु न है ओके हस् थ्याङ्क यू पख्नुस् एकैछिन है म एसक्यूएल बन्द गर्छु यसलाई मैले चलाइरहेको बन्द गर्छु है ए हस् हस् हुन्छ म्याम हस् थ्याङ्क यू हस् वेलकम"""

def clean_text_for_comparison(text):
    """Removes punctuation and extra spaces for fair comparison."""
    text = re.sub(r'[।?,!.:;।\"\'\(\)\[\]]', '', text)
    return " ".join(text.split()).strip()

def benchmark_indicwav2vec():
    print(f"--- INITIALIZING INDICWAV2VEC (CPU) ---")

    print(f"Loading Model: {MODEL_ID}...")
    try:
        processor = Wav2Vec2Processor.from_pretrained(MODEL_ID)
        model = Wav2Vec2ForCTC.from_pretrained(MODEL_ID)
    except Exception as e:
        print(f"Error loading model: {e}")
        return


    print(f"Processing Audio: {AUDIO_FILE}...")
    speech, sr = librosa.load(AUDIO_FILE, sr=16000)
    audio_duration = len(speech) / sr

    print("Transcribing (Inference on CPU)...")
    start_time = time.time()

    input_values = processor(speech, sampling_rate=16000, return_tensors="pt").input_values

    with torch.no_grad():
        logits = model(input_values).logits

    predicted_ids = torch.argmax(logits, dim=-1)

    generated_text = processor.batch_decode(predicted_ids)[0]

    execution_time = time.time() - start_time


    ref_norm = clean_text_for_comparison(REFERENCE_TEXT)
    gen_norm = clean_text_for_comparison(generated_text)

    w_error = wer(ref_norm, gen_norm)
    c_error = cer(ref_norm, gen_norm)

    print("\n" + "="*45)
    print("INDICWAV2VEC CPU PERFORMANCE REPORT")
    print("="*45)
    print(f"Audio Duration      : {audio_duration:.2f} seconds")
    print(f"Total Time Taken    : {execution_time:.2f} seconds")
    print(f"Real-Time Factor    : {execution_time / audio_duration:.4f}")
    print(f"Processing Speed    : {audio_duration / execution_time:.2f}x faster than audio")
    print("-" * 45)
    print(f"Word Error Rate (WER): {w_error * 100:.2f}%")
    print(f"Char Error Rate (CER): {c_error * 100:.2f}%")
    print(f"Final Accuracy (CER) : {(1 - c_error) * 100:.2f}%")
    print("="*45)

    print("\n--- GENERATED TRANSCRIPTION ---")
    print(generated_text)

    print("\n--- REFERENCE (GROUND TRUTH) ---")
    print(REFERENCE_TEXT[:200] + "...")

if __name__ == "__main__":
    benchmark_indicwav2vec()

--- INITIALIZING INDICWAV2VEC (CPU) ---
Loading Model: sumanpaudel1997/nepali-asr-indicwav2vec...


preprocessor_config.json:   0%|          | 0.00/256 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.15k [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 80), got 81. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 80), got 82. This may result in unexpected behavior.


tokenizer_config.json:   0%|          | 0.00/1.13k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  378MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/213 [00:00<?, ?it/s]

Processing Audio: out-0015592831-5053-20260802-125454-1785654594.80151.wav...
Transcribing (Inference on CPU)...

INDICWAV2VEC CPU PERFORMANCE REPORT
Audio Duration      : 157.10 seconds
Total Time Taken    : 104.10 seconds
Real-Time Factor    : 0.6626
Processing Speed    : 1.51x faster than audio
---------------------------------------------
Word Error Rate (WER): 94.69%
Char Error Rate (CER): 74.89%
Final Accuracy (CER) : 25.11%

--- GENERATED TRANSCRIPTION ---
शुकार्यबाट भोल्नुएकोैमले को चलउँदै थिए ले यप गर्दैन मर कामसके अअहिलेको योबन चेयक गर्नु भनिएन लसियमा लगिनु र्न मलाई छेन देखाउनकोलय  लाय लगनलािय लनत्र टाउ गर्न ले ईसो ईसिय देहाउ को होअब हुनो२्रब दरनय अ १टि टु १ति थ्री तेहा छैनएतल तल गर्नु देहा छहोला१ख होअ३२ु १ति थ्री मा जानु अनि लगिनु गर्नु अब लगिन१ैमले  यस केल बन्द गर्छअअर के प्रवलम आए भनि कल गर्नु नए१ यस केल भन्

--- REFERENCE (GROUND TRUTH) ---
हेल्लो हजुर सुब्बाकार्यबाट बोल्नुभएको हो नि म्याम हैन हजुर हजुर एकछिन मैले अहिले चलाउँदै थिएँ एकछिन एक्सेप्ट गर्दिनु न है मेरो काम सक